In [1]:
import os
import numpy as np
import pandas as pd

# === CONFIGURAÇÕES ===
base_path = "/media/fernando/KINGSTON/Doutorado/2025/reduced"
date_folder = "20250701"
export_path = os.path.join(base_path, date_folder, "export_txt")

output_suffix = "_diffphot.txt"

if not os.path.exists(export_path):
    raise FileNotFoundError(f"Pasta não encontrada: {export_path}")

# Arquivos .txt que NÃO terminam com polar.txt
txt_files = [
    f for f in os.listdir(export_path)
    if f.endswith(".txt") and not f.endswith("polar.txt")
]

print(f"{len(txt_files)} arquivo(s) encontrado(s).")

# === PROCESSAMENTO ===
for fname in txt_files:
    file_path = os.path.join(export_path, fname)
    print(f"\nProcessando: {fname}")

    try:
        df = pd.read_csv(file_path, delim_whitespace=True)

        mag_cols  = [c for c in df.columns if c.startswith("MAG_")]
        emag_cols = [c for c in df.columns if c.startswith("EMAG_")]
        src_cols  = [c for c in df.columns if c.startswith("SRCINDEX_")]

        # Alvo
        mag_target  = mag_cols[0]
        emag_target = emag_cols[0]
        src_target  = src_cols[0]

        # Comparação (3 estrelas)
        mag_comp  = mag_cols[1:4]
        emag_comp = emag_cols[1:4]

        # === FUNÇÃO AUXILIAR: mag -> fluxo + erro ===
        def mag_to_flux_and_error(m, em):
            F = 10**(-0.4 * m)
            F_plus  = 10**(-0.4 * (m - em))
            F_minus = 10**(-0.4 * (m + em))
            sigmaF = 0.5 * (F_plus - F_minus)
            return F, sigmaF

        # === ALVO ===
        F_target, sigmaF_target = mag_to_flux_and_error(
            df[mag_target], df[emag_target]
        )

        # === COMPARAÇÃO ===
        F_comp = np.zeros(len(df))
        sigmaF_comp_sq = np.zeros(len(df))

        for mcol, ecol in zip(mag_comp, emag_comp):
            F, sigmaF = mag_to_flux_and_error(df[mcol], df[ecol])
            F_comp += F
            sigmaF_comp_sq += sigmaF**2

        # Média dos fluxos
        N = len(mag_comp)
        F_comp /= N
        sigmaF_comp = np.sqrt(sigmaF_comp_sq) / N

        # === FOTOMETRIA DIFERENCIAL EM FLUXO ===
        F_diff = F_target / F_comp
        sigmaF_diff = F_diff * np.sqrt(
            (sigmaF_target / F_target)**2 +
            (sigmaF_comp   / F_comp)**2
        )

        # === CONVERSÃO FINAL PARA MAGNITUDE ===
        diff_mag = -2.5 * np.log10(F_diff)

        # erro em magnitude via limites (consistente)
        Fp = F_diff + sigmaF_diff
        Fm = F_diff - sigmaF_diff
        diff_emag = 0.5 * (
            -2.5 * np.log10(Fm) + 2.5 * np.log10(Fp)
        )

        # === SAÍDA ===
        out_df = pd.DataFrame({
            "TIME": df["TIME"],
            "SRCINDEX": df[src_target],
            "DIFF_MAG": diff_mag,
            "EDIFF_MAG": diff_emag#,
#            "EMAG": df[emag_target]
        })

        out_name = fname.replace(".txt", output_suffix)
        out_path = os.path.join(export_path, out_name)

        out_df.to_csv(
            out_path,
            sep="\t",
            index=False,
            float_format="%.8f"
        )

        print(f"[OK] Arquivo salvo: {out_name}")

    except Exception as e:
        print(f"[ERRO] {fname}: {e}")

24 arquivo(s) encontrado(s).

Processando: V2400_Oph_ch1_AP005_SRC5.txt
[OK] Arquivo salvo: V2400_Oph_ch1_AP005_SRC5_diffphot.txt

Processando: V2400_Oph_ch2_AP005_SRC13.txt
[OK] Arquivo salvo: V2400_Oph_ch2_AP005_SRC13_diffphot.txt

Processando: V2400_Oph_ch3_AP005_SRC21.txt
[OK] Arquivo salvo: V2400_Oph_ch3_AP005_SRC21_diffphot.txt

Processando: V2400_Oph_ch4_AP005_SRC35.txt
[OK] Arquivo salvo: V2400_Oph_ch4_AP005_SRC35_diffphot.txt

Processando: V2400_Oph_ch1_AP008_SRC5.txt
[OK] Arquivo salvo: V2400_Oph_ch1_AP008_SRC5_diffphot.txt

Processando: V2400_Oph_ch2_AP008_SRC13.txt
[OK] Arquivo salvo: V2400_Oph_ch2_AP008_SRC13_diffphot.txt

Processando: V2400_Oph_ch3_AP008_SRC21.txt
[OK] Arquivo salvo: V2400_Oph_ch3_AP008_SRC21_diffphot.txt

Processando: V2400_Oph_ch4_AP008_SRC35.txt
[OK] Arquivo salvo: V2400_Oph_ch4_AP008_SRC35_diffphot.txt

Processando: V2400_Oph_ch1_AP010_SRC5.txt
[OK] Arquivo salvo: V2400_Oph_ch1_AP010_SRC5_diffphot.txt

Processando: V2400_Oph_ch2_AP010_SRC13.txt
[OK] A

/tmp/ipykernel_14567/3532039511.py:29: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(file_path, delim_whitespace=True)
/tmp/ipykernel_14567/3532039511.py:29: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(file_path, delim_whitespace=True)
/tmp/ipykernel_14567/3532039511.py:29: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(file_path, delim_whitespace=True)
/tmp/ipykernel_14567/3532039511.py:29: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(file_path, delim_whitespace=True)
/tmp/ipykernel_14567/3532039511.py:29: FutureWarning: The 'delim